# Data Cleaning — `Tourist_Accommodation` (revisión 13/07/2026)

Revisión y actualización del notebook de limpieza `Data_Cleaning_06_07_2026.ipynb`.

Se mantiene toda la lógica de limpieza ya validada y **solo se añade/ajusta lo estrictamente
necesario** para que el notebook siga funcionando aunque cambie el fichero crudo de una semana
a otra (cambia el nombre, las columnas calculadas que trae, o incluso aparecen filas con
errores de formato). Cambios relevantes que motivan esta revisión:

- El fichero crudo puede traer o no, ya calculadas, las columnas de ocupación
  (`occupancy_30/60/90/365` y sus tasas), `rating_above_80` e `is_instant_bookable_numeric`.
  El notebook comprueba si están presentes antes de usarlas (sección 0).
- El formato de `insert_date` puede ser `dd/mm/aaaa` o `aaaa-mm-dd` según la semana; se usa una
  función que detecta el formato de cada valor en vez de asumir uno fijo (sección 1).
- Alguna fila puede llegar con las columnas desplazadas por un error de comillas sin cerrar en
  un campo de texto; se detectan y se descartan explícitamente antes de limpiar (ver más abajo).

**Puntos que se cubren (según instrucciones de Data Cleaning):**
1. Corrección de tipos de datos
2. Eliminación o corrección de duplicados
3. Tratamiento de valores faltantes
4. Validación y corrección de valores atípicos
5. Estandarización de formatos

**Salida final:** un único CSV limpio (`../Data/clean_dataset_13_07_2026.csv`) sobre el que se
aplicará la fase de *Data Transformation*.

> **Alcance de esta fase.** Aquí solo se limpia y se corrigen tipos/formatos. **NO** se hacen
> aquí (van en *Transformation*): creación de variables derivadas de negocio, codificación
> categórica definitiva, ni reducción de dimensionalidad.
> No se elimina información salvo lo estrictamente necesario (versiones antiguas de un mismo
> alojamiento, o filas irrecuperables por errores de formato).

In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)

In [5]:
# Nombre del fichero: única línea a cambiar cuando llegue un dataset nuevo
DATASET_FILE = 'raw_dataset_13_07_2026.csv'

def encontrar_raiz_proyecto(nombre_carpeta='Equip_34'):
    '''
    Función para encontrar la carpeta raíz del proyecto subiendo desde el directorio actual.
    '''
    actual = Path.cwd()
    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta
    raise FileNotFoundError(f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}")

raiz_proyecto = encontrar_raiz_proyecto('Equip_34')
ruta = raiz_proyecto / 'Data' / DATASET_FILE

print(f"Ruta resuelta: {ruta}")
df_original = pd.read_csv(ruta)
df = df_original.copy()

print("Dimensiones del crudo:", df.shape)

Ruta resuelta: c:\Users\giorg\Desktop\ProjecteData\Equip_34\Data\raw_dataset_13_07_2026.csv
Dimensiones del crudo: (10000, 35)


In [6]:
# Algunas filas pueden llegar "desplazadas": un error de comillas sin cerrar
# en un campo de texto (name/description/amenities_list) hace que el resto de
# columnas de esa fila se lean corridas, y columnas que deberían ser numéricas
# quedan con texto que no es un número. Se detectan así, no por texto exacto,
# para que el chequeo siga funcionando aunque cambien los datos.
cols_siempre_numericas = ['accommodates', 'availability_30', 'availability_60',
                           'availability_90', 'availability_365']

filas_desplazadas = pd.Series(False, index=df.index)
for col in cols_siempre_numericas:
    no_convierte = pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()
    filas_desplazadas |= no_convierte

print("Filas con columnas desplazadas detectadas:", int(filas_desplazadas.sum()))
if filas_desplazadas.sum() > 0:
    print(df.loc[filas_desplazadas, ['apartment_id', 'name']])

# Copia de trabajo, ya sin las filas desplazadas (no son recuperables de forma fiable)
df_clean = df[~filas_desplazadas].copy()
print("Dimensiones tras descartar filas desplazadas:", df_clean.shape)

Filas con columnas desplazadas detectadas: 0
Dimensiones tras descartar filas desplazadas: (10000, 35)


## 0. Columnas de ocupación y otras columnas calculadas en origen

Según la semana, el fichero crudo puede traer ya calculadas las columnas de ocupación
(`occupancy_30/60/90/365`, `occupancy_rate_30/60/90/365`), `rating_above_80` y
`is_instant_bookable_numeric`, o puede no traerlas (como esta semana). Esta sección comprueba
si están presentes y, solo si lo están, valida su coherencia. **`occupancy_365` es la variable
objetivo acordada con Verónica** para la pregunta de operaciones (habitaciones/baños/camas vs.
ocupación); si no viene en el crudo, hay que confirmar con el equipo si se recalculará en
Transformation o si debería llegar en el próximo volcado.

In [7]:
columnas_ocupacion = [f'occupancy_{p}' for p in [30, 60, 90, 365]]

if all(col in df_clean.columns for col in columnas_ocupacion):
    # occupancy_x + availability_x debe ser siempre igual al plazo (30/60/90/365)
    for periodo in [30, 60, 90, 365]:
        suma = df_clean[f'occupancy_{periodo}'] + df_clean[f'availability_{periodo}']
        print(f"occupancy_{periodo} + availability_{periodo} -> valores únicos: {suma.unique()}")

    # occupancy_rate_x debe ser coherente con occupancy_x / periodo * 100
    print()
    for periodo in [30, 60, 90, 365]:
        calc = (df_clean[f'occupancy_{periodo}'] / periodo * 100).round(2)
        diff = (calc - df_clean[f'occupancy_rate_{periodo}']).abs()
        print(f"occupancy_rate_{periodo}: diferencia máxima vs. cálculo propio = {diff.max()}")

    print("\nNulos en columnas de ocupación:")
    print(df_clean[columnas_ocupacion].isna().sum())
else:
    print("Este dataset NO incluye columnas de ocupación (occupancy_*).")
    print("Pendiente confirmar con Verónica/equipo cómo se obtiene occupancy_365 esta semana.")

Este dataset NO incluye columnas de ocupación (occupancy_*).
Pendiente confirmar con Verónica/equipo cómo se obtiene occupancy_365 esta semana.


In [8]:
if 'is_instant_bookable_numeric' in df_clean.columns:
    # is_instant_bookable_numeric: debe coincidir con el mapeo de is_instant_bookable
    chk = df_clean['is_instant_bookable'].map({'VERDADERO': 1, 'FALSO': 0})
    print("is_instant_bookable_numeric, discrepancias:", (chk != df_clean['is_instant_bookable_numeric']).sum())
else:
    print("Este dataset no incluye is_instant_bookable_numeric (se generará en Transformation).")

print(df_clean['is_instant_bookable'].value_counts(dropna=False))

Este dataset no incluye is_instant_bookable_numeric (se generará en Transformation).
is_instant_bookable
VERDADERO    5806
FALSO        4194
Name: count, dtype: int64


Cuando estas columnas están presentes llegan coherentes y completas (0 nulos, 0
discrepancias) y solo queda revisar su tipo de dato definitivo (sección 1). Cuando no están
presentes, quedan pendientes de Transformation o de una confirmación del equipo.

## 1. Corrección de tipos de datos

- `bathrooms`, `bedrooms` y `beds` llegan como `float64` aunque son recuentos → a entero
  nullable (`Int64`, admite nulos).
- `first_review_date`, `last_review_date` e `insert_date` son fechas en texto, pero **no
  siempre en el mismo formato**: puede ser `dd/mm/aaaa` o `aaaa-mm-dd` según la columna, y
  según nos recuerda Data Understanding, podría no ser siempre el mismo dentro de la propia
  columna en próximas entregas del dataset. Por eso se usa una función que detecta el formato
  de cada valor y lo convierte a `datetime` en lugar de asumir un único formato fijo (así el
  notebook sigue funcionando aunque el formato cambie de nuevo).
- `rating_above_80` llega como texto (`'True'`/`'False'`) → a booleano nullable (`boolean`).

In [9]:
# --- Recuentos que venían como texto/float -> entero nullable (Int64) ---
count_cols = ['bathrooms', 'bedrooms', 'beds']
for col in count_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')

# --- Fechas: admite dd/mm/aaaa o aaaa-mm-dd, detectando el formato de cada valor ---
def convertir_fecha(serie):
    '''Convierte una columna de fechas en texto a datetime, admitiendo
    los formatos dd/mm/aaaa y aaaa-mm-dd (pueden convivir en el dataset).'''
    texto = serie.astype('string')
    es_iso = texto.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)

    resultado = pd.Series(pd.NaT, index=serie.index, dtype='datetime64[us]')
    resultado.loc[es_iso] = pd.to_datetime(texto.loc[es_iso], format='%Y-%m-%d', errors='coerce')
    resultado.loc[~es_iso] = pd.to_datetime(texto.loc[~es_iso], format='%d/%m/%Y', errors='coerce')
    return resultado

date_cols = ['first_review_date', 'last_review_date', 'insert_date']
for col in date_cols:
    df_clean[col] = convertir_fecha(df_clean[col])

df_clean[count_cols + date_cols].dtypes

bathrooms                     Int64
bedrooms                      Int64
beds                          Int64
first_review_date    datetime64[us]
last_review_date     datetime64[us]
insert_date          datetime64[us]
dtype: object

In [10]:
# Comprobación: la conversión no debe crear nulos "nuevos" inesperados
print("Nulos tras convertir tipos (deben coincidir con los originales):")
for col in count_cols + date_cols:
    print(f"  {col:20s} nulos: {df_clean[col].isna().sum():5d}  |  original: {df[col].isna().sum()}")

Nulos tras convertir tipos (deben coincidir con los originales):
  bathrooms            nulos:    74  |  original: 74
  bedrooms             nulos:    70  |  original: 70
  beds                 nulos:    45  |  original: 45
  first_review_date    nulos:  2604  |  original: 2604
  last_review_date     nulos:  2605  |  original: 2605
  insert_date          nulos:     0  |  original: 0


In [11]:
if 'rating_above_80' in df_clean.columns:
    # rating_above_80: texto 'True'/'False' -> booleano nullable
    df_clean['rating_above_80'] = (
        df_clean['rating_above_80']
        .map({'True': True, 'False': False, True: True, False: False})
        .astype('boolean')
    )
    print(df_clean['rating_above_80'].dtype)
    print(df_clean['rating_above_80'].value_counts(dropna=False))
else:
    print("Este dataset no incluye rating_above_80 (se generará en Transformation).")

Este dataset no incluye rating_above_80 (se generará en Transformation).


## 2. Eliminación o corrección de duplicados

No hay filas completamente idénticas, pero sí `apartment_id` repetidos: el mismo alojamiento
volcado en fechas distintas. Se conserva únicamente la **última foto** (máximo `insert_date`) de
cada `apartment_id`. La lógica es idéntica a la semana pasada; el resultado (307 registros
eliminados) coincide porque el contenido de fondo es el mismo, solo cambia el formato de
`insert_date`, ya corregido en la sección anterior.

In [12]:
# Filas 100% duplicadas
print("Filas completamente duplicadas:", df_clean.duplicated().sum())

# apartment_id repetidos (mismo alojamiento en distintos volcados)
rep = df_clean['apartment_id'].value_counts()
rep = rep[rep > 1]
print(f"apartment_id repetidos: {rep.shape[0]}  (filas implicadas: {int(rep.sum())})")
print("Fecha global mas reciente de volcado (insert_date):", df_clean['insert_date'].max().date())

# Nos quedamos con el registro de mayor insert_date por apartment_id
antes = len(df_clean)
df_clean = (
    df_clean.sort_values('insert_date')
            .drop_duplicates(subset='apartment_id', keep='last')
            .reset_index(drop=True)
)
print(f"\nRegistros eliminados (versiones antiguas): {antes - len(df_clean)}")
print("Dimensiones tras deduplicar:", df_clean.shape)
print("apartment_id unico ahora?", df_clean['apartment_id'].is_unique)

Filas completamente duplicadas: 0
apartment_id repetidos: 342  (filas implicadas: 692)
Fecha global mas reciente de volcado (insert_date): 2021-02-27

Registros eliminados (versiones antiguas): 350
Dimensiones tras deduplicar: (9650, 35)
apartment_id unico ahora? True


## 3. Tratamiento de valores faltantes

Se **conservan** los registros con nulos (no se eliminan) y **no se imputan** valores que
puedan sesgar los análisis posteriores:

- `review_scores_*`, `rating_above_80` (si está presente), `first_review_date`,
  `last_review_date`, `reviews_per_month`: nulos **estructurales** = alojamientos sin reseñas
  (`number_of_reviews = 0`). Se dejan como nulos.
- `has_availability`: solo `VERDADERO` + nulos (no hay `FALSO`); los nulos se dejan como
  desconocido (no se convierten a falso). Se crea `has_availability_numeric` con valores 1 y 0
  respetando los nulos (si el crudo ya la trae calculada, solo se verifica que coincide con el
  mapeo directo).
- `neighbourhood_district`, `price`, `bathrooms`, `bedrooms`, `beds`, `name`, `description`:
  se conservan sin imputar.
- `occupancy_*` / `occupancy_rate_*` / `is_instant_bookable_numeric`: cuando están presentes,
  no tienen nulos (verificado en la sección 0) y no requieren tratamiento.

In [13]:
# Panorama de nulos tras deduplicar
nulos = pd.DataFrame({
    "nulos": df_clean.isnull().sum(),
    "%": (df_clean.isnull().mean() * 100).round(2)
})
nulos[nulos["nulos"] > 0].sort_values("%", ascending=False)

,nulos,%
neighbourhood_district,3790,39.27
review_scores_value,2640,27.36
review_scores_location,2640,27.36
review_scores_checkin,2639,27.35
review_scores_accuracy,2634,27.30
review_scores_communication,2630,27.25
review_scores_cleanliness,2628,27.23
review_scores_rating,2625,27.20
last_review_date,2523,26.15
first_review_date,2522,26.13


In [14]:
# Los nulos de valoraciones son estructurales: alojamientos sin reseñas
sin_reviews = df_clean['number_of_reviews'] == 0
print("Alojamientos con number_of_reviews == 0 :", sin_reviews.sum())
print("  de ellos con review_scores_rating nulo:", (sin_reviews & df_clean['review_scores_rating'].isna()).sum())
print("Con reseñas (>0) pero rating nulo (nulo real, no estructural):",
      ((~sin_reviews) & df_clean['review_scores_rating'].isna()).sum())

if 'rating_above_80' in df_clean.columns:
    print("rating_above_80 nulo coincide exactamente con review_scores_rating nulo:",
          (df_clean['rating_above_80'].isna() == df_clean['review_scores_rating'].isna()).all())

print("\nhas_availability (solo VERDADERO + nulos, sin FALSO):")
print(df_clean['has_availability'].value_counts(dropna=False))

Alojamientos con number_of_reviews == 0 : 2519
  de ellos con review_scores_rating nulo: 2519
Con reseñas (>0) pero rating nulo (nulo real, no estructural): 106

has_availability (solo VERDADERO + nulos, sin FALSO):
has_availability
VERDADERO    9116
NaN           534
Name: count, dtype: int64


In [15]:
# has_availability_numeric: se crea si no existe; si ya viene calculada, se verifica
mapeo_directo = df_clean['has_availability'].map({'VERDADERO': 1, 'FALSO': 0}).astype('Int64')

if 'has_availability_numeric' in df_clean.columns:
    discrepancias = (mapeo_directo != df_clean['has_availability_numeric'])
    discrepancias = discrepancias & ~(mapeo_directo.isna() & df_clean['has_availability_numeric'].isna())
    print("Discrepancias has_availability_numeric vs mapeo directo:", int(discrepancias.sum()))
else:
    df_clean['has_availability_numeric'] = mapeo_directo
    print("has_availability_numeric no venía en el crudo: se ha creado con el mapeo directo.")

print(df_clean['has_availability_numeric'].value_counts(dropna=False))
df_clean[['has_availability', 'has_availability_numeric']].tail()

has_availability_numeric no venía en el crudo: se ha creado con el mapeo directo.
has_availability_numeric
1       9116
<NA>     534
Name: count, dtype: Int64


,has_availability,has_availability_numeric
9645,VERDADERO,1
9646,VERDADERO,1
9647,VERDADERO,1
9648,VERDADERO,1
9649,VERDADERO,1


In [16]:
df_clean["reviews_per_month"].describe()

count    7128.000000
mean      123.422559
std       152.705821
min         1.000000
25%        18.000000
50%        58.000000
75%       177.000000
max      1273.000000
Name: reviews_per_month, dtype: float64

In [17]:
df_clean["reviews_per_month"].sort_values(ascending=False).head(10)

4996    1273.0
2359    1213.0
6997    1125.0
7271    1045.0
5319     991.0
4326     952.0
987      938.0
7536     927.0
4259     914.0
4199     911.0
Name: reviews_per_month, dtype: float64

## 4. Validación y corrección de valores atípicos

Confirmamos que los valores extremos son plausibles (no errores) y los **mantenemos**. Solo se
revisa coherencia; no se elimina nada.

> `reviews_per_month` presenta valores muy altos (máx ≈ 1273). Parece estar escalado (×100)
> respecto al valor mensual real; **no se corrige aquí**, se revisará en *Transformation*.

> Las columnas de ocupación (`occupancy_365`, etc.) ya se validaron en la sección 0: son
> complementarias exactas de `availability_x` sobre el mismo plazo, así que no pueden tener
> valores fuera de rango (`[0, plazo]`) por construcción.

In [18]:
num_check = ['price', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
             'minimum_nights', 'maximum_nights', 'number_of_reviews', 'reviews_per_month']
df_clean[num_check].describe().round(2)

,price,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,number_of_reviews,reviews_per_month
count,9409.00,9650.00,9578.0,9580.0,9605.0,9650.00,9650.00,9650.00,7128.00
mean,1019.64,4.27,1.59,1.94,2.92,4.97,758.33,25.97,123.42
std,977.49,2.59,0.99,1.38,2.27,17.99,498.81,52.51,152.71
min,60.00,1.00,0.0,0.0,0.0,1.00,1.00,0.00,1.00
25%,450.00,2.00,1.0,1.0,1.0,1.00,61.00,0.00,18.00
50%,750.00,4.00,1.0,2.0,2.0,2.00,1125.00,5.00,58.00
75%,1240.00,6.00,2.0,3.0,4.0,4.00,1125.00,27.00,177.00
max,28571.00,29.00,13.0,50.0,30.0,1125.00,1125.00,588.00,1273.00


In [19]:
# Recuento de outliers por IQR (solo informativo: NO se eliminan)
def n_outliers_iqr(s):
    s = s.dropna().astype(float)
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < low) | (s > high)).sum())

resumen_out = pd.Series({c: n_outliers_iqr(df_clean[c]) for c in num_check}, name='outliers_IQR')
print(resumen_out.to_string())

# Casos limite coherentes que SE CONSERVAN (pueden aportar info en otras columnas)
print("\nAlojamientos con beds == 0      :", int((df_clean['beds'] == 0).sum()), "(se conservan)")
print("Alojamientos con bathrooms == 0 :", int((df_clean['bathrooms'] == 0).sum()), "(se conservan)")
print("price -> min:", df_clean['price'].min(), "| max:", df_clean['price'].max(), "| nulos:", int(df_clean['price'].isna().sum()))

price                 718
accommodates           81
bathrooms             451
bedrooms               48
beds                  214
minimum_nights        622
maximum_nights          0
number_of_reviews    1120
reviews_per_month     454

Alojamientos con beds == 0      : 121 (se conservan)
Alojamientos con bathrooms == 0 : 21 (se conservan)
price -> min: 60.0 | max: 28571.0 | nulos: 241


In [20]:
if 'occupancy_365' in df_clean.columns:
    # occupancy_365 fuera de rango [0, 365]? (comprobación de sanidad para la pregunta de operaciones)
    fuera_rango = ((df_clean['occupancy_365'] < 0) | (df_clean['occupancy_365'] > 365)).sum()
    print("occupancy_365 fuera de [0, 365]:", fuera_rango)
    print(df_clean['occupancy_365'].describe())
else:
    print("occupancy_365 no está en este dataset: no se puede comprobar (ver sección 0).")

occupancy_365 no está en este dataset: no se puede comprobar (ver sección 0).


## 5. Estandarización de formatos

- Espacios sobrantes en columnas de texto.
- `price` → `price_€` (se mantiene numérico, solo cambia el nombre para dejar explícita la
  unidad, como en la semana pasada).
- `city`: **en el fichero nuevo llega en minúsculas** (`barcelona`, `madrid`...) en vez de
  mixto como antes; se normaliza igual que la semana pasada (primera letra en mayúscula), y el
  resultado es equivalente.
- Limpieza de `amenities_list` (codificación, sinónimos, tokens de traducción faltante).

In [21]:
# Espacios sobrantes en columnas de texto
text_cols = df_clean.select_dtypes(include=['object']).columns
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# price -> price_€ (se mantiene numerico)
df_clean = df_clean.rename(columns={'price': 'price_€'})
print("Columna de precio:", [c for c in df_clean.columns if 'price' in c], "| dtype:", df_clean['price_€'].dtype)

# Columna `city` pasa a ser mayuscula la primera letra
print("Valores de city antes de normalizar:", sorted(df_clean['city'].str.strip().unique()))
df_clean["city"] = df_clean["city"].str.strip().str.capitalize()
print("\ncolumna City normalizada:")
print(df_clean['city'].value_counts())

Columna de precio: ['price_€'] | dtype: float64
Valores de city antes de normalizar: ['barcelona', 'girona', 'madrid', 'malaga', 'mallorca', 'menorca', 'sevilla', 'valencia']

columna City normalizada:
city
Barcelona    2719
Madrid       2134
Mallorca     1583
Girona       1485
Valencia      513
Malaga        504
Sevilla       494
Menorca       218
Name: count, dtype: int64


In [22]:
# Cuantificacion del problema de codificacion (informativo, no se altera el dato)
mojibake_cols = ['name', 'neighbourhood_name', 'description']
for col in mojibake_cols:
    n = df_clean[col].fillna('').str.contains('�').sum()
    print(f"{col:20s}: {n} registros con caracter de codificacion perdido (acento perdido en origen)")

name                : 2078 registros con caracter de codificacion perdido (acento perdido en origen)
neighbourhood_name  : 2110 registros con caracter de codificacion perdido (acento perdido en origen)
description         : 6629 registros con caracter de codificacion perdido (acento perdido en origen)


In [23]:
# Tratamiento de translation missing: en.hosting_amenity_49/50
print(
    "Registros afectados:",
    df_clean["amenities_list"].str.contains(
        "translation missing",
        case=False,
        na=False
    ).sum()
)

Registros afectados: 927


In [24]:
# identificamos la existencia de los valores erroneos
(df_clean.loc[
        df_clean["amenities_list"].str.contains(
            "translation missing",
            case=False,
            na=False
        ),
        "amenities_list"
    ]
    .str.extractall(r"(translation missing:[^,\]]+)")
    .value_counts())

0                                         
translation missing: en.hosting_amenity_50    841
translation missing: en.hosting_amenity_49    633
Name: count, dtype: int64

In [25]:
# eliminamos los valores erroneos
valores_eliminar = [
    "translation missing: en.hosting_amenity_49",
    "translation missing: en.hosting_amenity_50"
]

for token in valores_eliminar:
    df_clean["amenities_list"] = (
        df_clean["amenities_list"]
        .str.replace(token, "", regex=False)
    )

In [26]:
#eliminamos posibles , dobles que puedan quedar de la eliminacion
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
        .str.replace(r"\s*,\s*,", ",", regex=True)
        .str.replace(r"^,\s*", "", regex=True)
        .str.replace(r",\s*$", "", regex=True)
        .str.replace(r"\s{2,}", " ", regex=True)
        .str.strip()
)

In [27]:
#en los casos en que quede vacia la lista, se convierten a nulos
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
        .replace(r"^\s*$", pd.NA, regex=True)
)

In [28]:
amenities = (
    df_clean["amenities_list"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
)

In [29]:
# confirmamos que no hay registros vacios
print("Amenities vacías:", (amenities == "").sum())
print("Amenities únicas:", amenities.nunique())

Amenities vacías: 295
Amenities únicas: 357


In [30]:
# Unificación del nombre de ciudad (por si quedara alguna variante suelta)
df_clean["city"] = df_clean["city"].replace({
    "Malaga": "Málaga"
})

Se detectaron registros donde la columna `amenities_list` contenía una cadena vacía
(`''`) en lugar de un valor nulo. Se estandarizaron a `NaN` para representar la ausencia de
información y evitar generar una comodidad vacía durante la separación de la lista.

### Estandarización de sinónimos y codificación en `amenities_list`

In [31]:
# Obtener todas las amenities únicas
amenities = (
    df_clean["amenities_list"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

amenities.sort_values().unique()

array(['', '24-hour check-in', '40 HDTV', '43 HDTV with Netflix',
       'Accessible-height bed', 'Accessible-height toilet',
       'Air conditioning', 'Air conditioning]', 'Amazon Echo',
       'BBQ grill', 'BBQ grill]', 'Baby bath', 'Baby monitor',
       'Baby safety gates', 'Baby safety gates]',
       'Babysitter recommendations', 'Baking sheet', 'Balcony',
       'Barbecue utensils', 'Bath towel', 'Bathroom essentials',
       'Bathtub', 'Bathtub with bath chair', 'Bathtub]',
       'Beach essentials', 'Beach essentials]', 'Beach view',
       'Beachfront', 'Beachfront]', 'Bed linens', 'Bed linens]',
       'Bedroom comforts', 'Bedroom comforts]', 'Bidet',
       'Bluetooth sound system', 'Body soap', 'Bread maker', 'Breakfast',
       'Breakfast bar', 'Breakfast table', 'Breakfast]', 'Building staff',
       'Building staff]', 'Buzzer/wireless intercom', 'Cable TV',
       'Cable TV]', 'Carbon monoxide alarm', 'Carbon monoxide alarm]',
       'Carbon monoxide detector', 'Cat(s)

In [32]:
sinonimos = {

    # Sinonimos
    "Smoke detector": "Smoke alarm",
    "Carbon monoxide detector": "Carbon monoxide alarm",
    "Wireless Internet": "Wifi",

    # Capitalizacion
    "Self Check-In": "Self check-in",
    "Laptop-friendly workspace": "Laptop friendly workspace",
    "Doorman Entry": "Doorman",

    # Problemas de codificacion
    "Children\\u2019s books and toys": "Children\'s books and toys",
    "Children\ufffds books and toys": "Children\'s books and toys",
    "Childrenu2019s books and toys": "Children\'s books and toys",

    "Children\\u2019s dinnerware": "Children\'s dinnerware",
    "Children\ufffds dinnerware": "Children\'s dinnerware",
    "Childrenu2019s dinnerware": "Children\'s dinnerware",

    "Pack \\u2019n Play/travel crib": "Pack \'n Play/travel crib",
    "Pack \ufffdn Play/travel crib": "Pack \'n Play/travel crib",
    "Pack u2019n Play/travel crib": "Pack \'n Play/travel crib",

    "Wifi u2013 100 Mbps": "Wifi",
    "Washer u2013 In unit": "Washer",
    "Washer u2013u00a0In unit": "Washer",

    "Free driveway parking on premises u2013 1 space":
    "Free driveway parking on premises - 1 space",

    # Errores puntuales
    "43 HDTVwith Netflix": "43 HDTV with Netflix",
    "Pour Over Coffee": "Pour-over coffee",
    "Ski in/Ski out": "Ski-in/Ski-out",
    "smooth pathway to front door": "Smooth pathway to front door",
    "toilet": "Toilet"
}

for viejo, nuevo in sinonimos.items():
    df_clean["amenities_list"] = (
        df_clean["amenities_list"]
        .str.replace(viejo, nuevo, regex=False)
    )

In [33]:
# Eliminar espacios alrededor de las comas
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r"\s*,\s*", ",", regex=True)
)

# Eliminar comas repetidas
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r",\s*,+", ",", regex=True)
)

# Eliminar coma final
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace(r",\s*$", "", regex=True)
    .str.strip()
)

In [34]:
amenities = (
    df_clean["amenities_list"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# No deben quedar elementos vacios
print("Amenities vacías:", (amenities == "").sum())

# Numero de amenities distintas
print("Amenities únicas:", amenities.nunique())

# Frecuencias
amenities_df = (
    amenities
    .value_counts()
    .reset_index()
)

amenities_df.columns = ["Amenity", "Frecuencia"]

amenities_df

Amenities vacías: 0
Amenities únicas: 343


,Amenity,Frecuencia
0,Kitchen,8764
1,Wifi,8710
2,Essentials,8503
3,Washer,7996
4,TV,7948
...,...,...
338,Outdoor dining area,1
339,Outdoor shower,1
340,Paid parking garage on premises,1
341,Stainless steel stove,1


In [35]:
#convertimos las cadenas vacias a nulos
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .replace(r"^\s*$", pd.NA, regex=True)
)

In [36]:
amenities = (
    df_clean["amenities_list"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
)

In [37]:
# confirmamos que no hay registros vacios
print("Amenities vacías:", (amenities == "").sum())
print("Amenities únicas:", amenities.nunique())

Amenities vacías: 0
Amenities únicas: 343


In [38]:
# limpieza de formato. eliminacion de `]`
df_clean["amenities_list"] = (
    df_clean["amenities_list"]
    .str.replace("]", "", regex=False)
)

Se detectaron registros donde la columna `amenities_list` contenía una cadena vacía
(`''`) tras la limpieza de tokens y sinónimos. Se estandarizaron a `NaN` para representar la
ausencia de información y evitar generar una comodidad vacía durante la separación de la
lista.

## 6. Verificación final y variables de interés por departamento

Antes de exportar, se comprueba que están presentes las variables necesarias para las tres
preguntas de negocio.

Se genera un **único CSV** con el resultado de la limpieza, listo para la fase de
*Data Transformation*.

In [39]:
print("Dimensiones finales:", df_clean.shape)
print("apartment_id unico :", df_clean['apartment_id'].is_unique)
print("Filas duplicadas   :", df_clean.duplicated().sum())

variables_interes = {
    "Marketing": ['city', 'neighbourhood_name', 'review_scores_location',
                  'minimum_nights', 'maximum_nights'],
    "Operaciones": ['bedrooms', 'bathrooms', 'beds', 'occupancy_30',
                               'occupancy_60', 'occupancy_90', 'occupancy_365',
                               'is_instant_bookable', 'city'],
    "Experiencia cliente": ['price_€', 'review_scores_rating', 'city'],
}
print("\nComprobación de variables de interés por departamento:")
for depto, cols in variables_interes.items():
    faltan = [c for c in cols if c not in df_clean.columns]
    print(f"  {depto:25s} -> faltan: {faltan if faltan else 'ninguna'}")

print("\nTipos de datos finales:")
df_clean.info()

Dimensiones finales: (9650, 36)
apartment_id unico : True
Filas duplicadas   : 0

Comprobación de variables de interés por departamento:
  Marketing                 -> faltan: ninguna
  Operaciones               -> faltan: ['occupancy_30', 'occupancy_60', 'occupancy_90', 'occupancy_365']
  Experiencia cliente       -> faltan: ninguna

Tipos de datos finales:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 9650 non-null   int64         
 1   name                         9647 non-null   object        
 2   description                  9516 non-null   object        
 3   host_id                      9650 non-null   int64         
 4   neighbourhood_name           9650 non-null   object        
 5   neighbourhood_district       5860 non-null   object        
 6   room_ty

In [40]:
# Exportacion del CSV final (una sola tabla limpia) para Data Transformation
output_dir = Path('..') / 'Data'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'clean_dataset_13_07_2026.csv'

# utf-8 (sin BOM) para un round-trip limpio con pandas en la fase siguiente
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print("CSV limpio guardado en:", output_path.resolve())
print("Filas x columnas:", df_clean.shape)

CSV limpio guardado en: C:\Users\giorg\Desktop\ProjecteData\Equip_34\Data\clean_dataset_13_07_2026.csv
Filas x columnas: (9650, 36)
